# SharePoint Ingestion

This notebook extracts E-SMAT data from SharePoint Lists through Microsoft Graph API and writes validated raw datasets to the Bronze layer.

## Responsibilities

1. Load reusable functions;
2. Read protected API configuration;
3. Authenticate with Microsoft Graph API;
4. Define source-list and Bronze-table mappings;
5. Discover configured SharePoint Lists;
6. Extract all records using pagination;
7. Validate each source schema before DataFrame creation;
8. Standardize column names;
9. Add ingestion metadata;
10. Persist raw Delta tables.

## Outputs

- `inspection_header_bronze`
- `inspection_responses_bronze`
- `inspection_requirements_bronze`

> Production credentials, list names, field identifiers, catalog names, schema names, and storage paths are represented by placeholders in this portfolio version.


## 1. Load shared functions

The shared notebook contains reusable Graph API, SharePoint, schema-validation, and Spark helpers. It does not execute the ingestion pipeline by itself.


In [ ]:
%run ./01_shared_functions


## 2. Load protected configuration

The credentials file must be stored outside the Git repository. In production, use an approved secret-management mechanism or a protected Unity Catalog Volume.


In [ ]:
import json
from pathlib import Path

CONFIG_PATH = Path(
    "/Volumes/<catalog>/<schema>/<volume>/graph_api_config.json"
)

print("Loading Microsoft Graph API configuration...")

try:
    with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
        config = json.load(config_file)

    graph_config = config["graph_api"]
    tenant_id = graph_config["tenant_id"]
    client_id = graph_config["client_id"]
    client_secret = graph_config["client_secret"]
    site_id = graph_config["site_id"]

except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Configuration file not found: {CONFIG_PATH}"
    ) from exc
except KeyError as exc:
    raise KeyError(
        f"Missing required configuration key: {exc}"
    ) from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Invalid JSON configuration file: {CONFIG_PATH}"
    ) from exc

print("API configuration loaded successfully.")


## 3. Authenticate with Microsoft Graph API

The OAuth 2.0 client-credentials flow is used to obtain an application access token. The token and credentials are never displayed.


In [ ]:
TOKEN_URL = (
    f"https://login.microsoftonline.com/{tenant_id}"
    "/oauth2/v2.0/token"
)

TOKEN_PAYLOAD = {
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "https://graph.microsoft.com/.default",
    "grant_type": "client_credentials",
}

print("Requesting Microsoft Graph API access token...")

token_response = requests.post(
    TOKEN_URL,
    data=TOKEN_PAYLOAD,
    timeout=DEFAULT_REQUEST_TIMEOUT_SECONDS,
)

try:
    token_response.raise_for_status()
except requests.HTTPError as exc:
    raise RuntimeError(
        "Microsoft Graph authentication failed. "
        f"Status: {token_response.status_code}. "
        f"Response: {token_response.text[:1000]}"
    ) from exc

token_payload = token_response.json()
access_token = token_payload.get("access_token")

if not access_token:
    raise RuntimeError(
        "Microsoft Graph authentication response did not contain an access token."
    )

request_headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json",
}

print("Authentication completed successfully.")


## 4. Configure source lists and Bronze targets

The outer dictionary keys are stable analytical dataset names. `sharepoint_list_name` and optional internal-field aliases must be replaced only in the private production version.

The ingestion layer initially keeps source values as nullable strings. Explicit typing and business transformations are applied in the next notebook.


In [ ]:
BRONZE_CATALOG = "<catalog>"
BRONZE_SCHEMA = "<bronze-schema>"

LISTS_CONFIG = {
    "inspection_header": {
        "sharepoint_list_name": "<inspection-header-list>",
        "target_table": (
            f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.inspection_header_bronze"
        ),
        "columns": [
            "Title",
            "Site",
            "InspectionDate",
            "Shift",
            "Segment",
            "BusinessUnitArea",
            "Line",
            "Workstation",
            "ActivityType",
            "ContractType",
            "ObservedPersonId1",
            "JobRole",
            "ObserverId",
            "ObservedPersonId2",
            "ObservedPersonId3",
            "UnsafeCondition",
            "UnsafeConditionDescription",
            "SafetyTagOpened",
            "SafetyTagNumber",
            "NearMiss",
            "NearMissDescription",
            "AuditType",
            "Author",
        ],
        "rename_mapping": {
            "Title": "inspection_id",
            "Site": "site",
            "InspectionDate": "inspection_datetime",
            "Shift": "shift",
            "Segment": "segment",
            "BusinessUnitArea": "business_unit_area",
            "Line": "line",
            "Workstation": "workstation",
            "ActivityType": "activity_type",
            "ContractType": "contract_type",
            "ObservedPersonId1": "observed_person_id_1",
            "JobRole": "job_role",
            "ObserverId": "observer_id",
            "ObservedPersonId2": "observed_person_id_2",
            "ObservedPersonId3": "observed_person_id_3",
            "UnsafeCondition": "unsafe_condition",
            "UnsafeConditionDescription": "unsafe_condition_description",
            "SafetyTagOpened": "safety_tag_opened",
            "SafetyTagNumber": "safety_tag_number",
            "NearMiss": "near_miss",
            "NearMissDescription": "near_miss_description",
            "AuditType": "audit_type",
            "Author": "created_by",
        },
        "field_aliases": {},
    },
    "inspection_responses": {
        "sharepoint_list_name": "<inspection-responses-list>",
        "target_table": (
            f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.inspection_responses_bronze"
        ),
        "columns": [
            "Title",
            "Site",
            "QuestionId",
            "SafeAct",
            "UnsafeAct",
            "Description",
            "InspectionDate",
            "Author",
        ],
        "rename_mapping": {
            "Title": "inspection_id",
            "Site": "site",
            "QuestionId": "question_id",
            "SafeAct": "safe_act",
            "UnsafeAct": "unsafe_act",
            "Description": "description",
            "InspectionDate": "inspection_date",
            "Author": "created_by",
        },
        "field_aliases": {},
    },
    "inspection_requirements": {
        "sharepoint_list_name": "<inspection-requirements-list>",
        "target_table": (
            f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.inspection_requirements_bronze"
        ),
        "columns": [
            "Title",
            "Site",
            "Methodology",
            "QuestionnaireVersion",
            "Theme",
            "CategoryId",
            "CategoryNamePtBr",
            "CategoryNameEsAr",
            "QuestionId",
            "QuestionColumnId",
            "QuestionNumber",
            "QuestionDescription",
            "Active",
        ],
        "rename_mapping": {
            "Title": "record_title",
            "Site": "site",
            "Methodology": "methodology",
            "QuestionnaireVersion": "questionnaire_version",
            "Theme": "theme",
            "CategoryId": "category_id",
            "CategoryNamePtBr": "category_name_pt_br",
            "CategoryNameEsAr": "category_name_es_ar",
            "QuestionId": "question_id",
            "QuestionColumnId": "question_column_id",
            "QuestionNumber": "question_number",
            "QuestionDescription": "requirement_text",
            "Active": "active",
        },
        "field_aliases": {
            # "QuestionId": "<internal-sharepoint-field-name>",
        },
    },
}

WRITE_MODE = "overwrite"
OVERWRITE_SCHEMA = "true"


## 5. Validate notebook configuration

Configuration errors are detected before any SharePoint extraction begins.


In [ ]:
def validate_ingestion_config(lists_config):
    if not lists_config:
        raise ValueError("LISTS_CONFIG cannot be empty.")

    required_config_keys = {
        "sharepoint_list_name",
        "target_table",
        "columns",
        "rename_mapping",
        "field_aliases",
    }

    target_tables = []

    for dataset_name, dataset_config in lists_config.items():
        missing_keys = sorted(required_config_keys - set(dataset_config))
        if missing_keys:
            raise ValueError(
                f"Dataset '{dataset_name}' is missing configuration keys: "
                f"{missing_keys}"
            )

        source_columns = dataset_config["columns"]
        rename_mapping = dataset_config["rename_mapping"]
        field_aliases = dataset_config["field_aliases"]
        target_table = dataset_config["target_table"]

        if not source_columns:
            raise ValueError(
                f"Dataset '{dataset_name}' has no configured source columns."
            )
        if len(source_columns) != len(set(source_columns)):
            raise ValueError(
                f"Dataset '{dataset_name}' has duplicate source columns."
            )
        if set(rename_mapping) - set(source_columns):
            raise ValueError(
                f"Dataset '{dataset_name}' has mappings for unconfigured fields."
            )
        if set(field_aliases) - set(source_columns):
            raise ValueError(
                f"Dataset '{dataset_name}' has aliases for unconfigured fields."
            )
        if target_table.count(".") != 2:
            raise ValueError(
                f"Dataset '{dataset_name}' has an invalid target table: "
                f"{target_table}"
            )

        target_tables.append(target_table)

    if len(target_tables) != len(set(target_tables)):
        raise ValueError("Multiple datasets use the same Bronze target table.")

    print(
        f"Ingestion configuration validated for {len(lists_config)} datasets."
    )


validate_ingestion_config(LISTS_CONFIG)


## 6. Discover SharePoint Lists

The list-discovery endpoint is paginated. Configured list names are matched to Microsoft Graph list identifiers.


In [ ]:
lists_url = (
    f"{GRAPH_BASE_URL}/sites/{site_id}/lists"
    "?$select=id,displayName"
)

print("Discovering SharePoint Lists...")

discovered_lists = get_paginated_graph_values(
    initial_url=lists_url,
    request_headers=request_headers,
)

list_ids_by_name = {
    item["displayName"]: item["id"]
    for item in discovered_lists
    if item.get("displayName") and item.get("id")
}

configured_list_names = {
    dataset_name: dataset_config["sharepoint_list_name"]
    for dataset_name, dataset_config in LISTS_CONFIG.items()
}

missing_lists = {
    dataset_name: list_name
    for dataset_name, list_name in configured_list_names.items()
    if list_name not in list_ids_by_name
}

if missing_lists:
    print("Warning: some configured SharePoint Lists were not found:")
    for dataset_name, list_name in missing_lists.items():
        print(f"  - {dataset_name}: {list_name}")
else:
    print("All configured SharePoint Lists were found.")


## 7. Extract, validate, and create raw DataFrames

Every list is fully extracted through pagination. The transformed record schema is validated before Spark DataFrame creation, then validated again after column standardization.


In [ ]:
processed_dataframes = {}
extraction_summary = []

for dataset_name, dataset_config in LISTS_CONFIG.items():
    sharepoint_list_name = dataset_config["sharepoint_list_name"]
    list_id = list_ids_by_name.get(sharepoint_list_name)

    if not list_id:
        extraction_summary.append(
            {
                "dataset": dataset_name,
                "status": "not_found",
                "records": 0,
                "target_table": dataset_config["target_table"],
            }
        )
        continue

    print(f"Extracting dataset: {dataset_name}")

    items_url = (
        f"{GRAPH_BASE_URL}/sites/{site_id}/lists/{list_id}/items"
        "?$expand=fields"
    )

    sharepoint_items = get_paginated_graph_values(
        initial_url=items_url,
        request_headers=request_headers,
    )

    if not sharepoint_items:
        print(f"Dataset '{dataset_name}' is empty.")
        extraction_summary.append(
            {
                "dataset": dataset_name,
                "status": "empty",
                "records": 0,
                "target_table": dataset_config["target_table"],
            }
        )
        continue

    source_columns = dataset_config["columns"]
    rename_mapping = dataset_config["rename_mapping"]
    field_aliases = dataset_config["field_aliases"]

    records = transform_sharepoint_items(
        sharepoint_items=sharepoint_items,
        source_columns=source_columns,
        field_aliases=field_aliases,
    )

    validation_result = validate_records_schema(
        records=records,
        source_columns=source_columns,
        rename_mapping=rename_mapping,
        dataset_name=dataset_name,
    )

    dataframe = create_string_dataframe(
        spark_session=spark,
        records=records,
        source_columns=source_columns,
        dataset_name=dataset_name,
    )

    dataframe = apply_column_mapping(
        dataframe=dataframe,
        rename_mapping=rename_mapping,
    )

    dataframe = validate_renamed_dataframe(
        dataframe=dataframe,
        source_columns=source_columns,
        rename_mapping=rename_mapping,
        dataset_name=dataset_name,
    )

    dataframe = (
        add_processing_timestamp(dataframe)
        .withColumn("source_system", F.lit("sharepoint"))
        .withColumn("source_list", F.lit(sharepoint_list_name))
    )

    record_count = len(records)

    processed_dataframes[dataset_name] = {
        "df": dataframe,
        "target_table": dataset_config["target_table"],
        "source_list": sharepoint_list_name,
        "source_list_id": list_id,
        "record_count": record_count,
        "schema_validation": validation_result,
    }

    extraction_summary.append(
        {
            "dataset": dataset_name,
            "status": "processed",
            "records": record_count,
            "target_table": dataset_config["target_table"],
        }
    )

    print(
        f"Dataset '{dataset_name}' loaded in memory with "
        f"{record_count} records."
    )

if not processed_dataframes:
    raise RuntimeError(
        "No SharePoint dataset was available for Bronze persistence."
    )

print("SharePoint extraction and schema validation completed.")


## 8. Review extraction summary

In [ ]:
extraction_summary_df = spark.createDataFrame(extraction_summary)
display(extraction_summary_df.orderBy("dataset"))


## 9. Persist Bronze Delta tables

Bronze tables preserve standardized source values and ingestion metadata. Business transformations are intentionally deferred to the next notebook.


In [ ]:
load_summary = []

for dataset_name, dataset_info in processed_dataframes.items():
    dataframe = dataset_info["df"]
    target_table = dataset_info["target_table"]
    record_count = dataset_info["record_count"]

    require_columns(
        dataframe=dataframe,
        required_columns={
            "processing_timestamp",
            "source_system",
            "source_list",
        },
        dataset_name=dataset_name,
    )

    print(f"Writing Bronze dataset '{dataset_name}' to {target_table}...")

    (
        dataframe.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", OVERWRITE_SCHEMA)
        .saveAsTable(target_table)
    )

    load_summary.append(
        {
            "dataset": dataset_name,
            "target_table": target_table,
            "records": record_count,
            "status": "written",
        }
    )

    print(f"Bronze table written successfully: {record_count} records.")

bronze_load_summary_df = spark.createDataFrame(load_summary)
display(bronze_load_summary_df.orderBy("dataset"))

print("SharePoint Bronze ingestion completed successfully.")


## Next step

Run `03_data_transformation.ipynb` to read the Bronze tables, apply business transformations, normalize data types, enrich responses with safety/environment themes, and create Silver datasets.
